# Week 1 Coding (20 min)

Three tasks, each connected to the problem set: **C1** ↔ P3/P4, **C2** ↔ P2, **C3** ↔ P6/P10.

**Predict first.** Before running any check cell, write down on paper what the output will be.

**How to access and work in this notebook.** Access this notebook through the Moodle activity **“L01 - Post-class - Jupyter Notebook Link.”** The first time you access the exercise, click **“Get a copy of the assignment”** to create your personal copy in your ETH JupyterHub workspace. Then open the notebook through the same Moodle activity. If prompted, click **“Start My Server”** and wait for JupyterLab to load. Run cells top to bottom with Shift+Enter. Each task has a cell with gaps marked `...`: fill them, run the cell, then execute the check cell below, which compares the result with the corresponding problem. If the notebook gets into a confused state, use **Kernel → Restart Kernel**, then run the cells again from the top. When returning to the exercise later, access your notebook again through the same Moodle activity. The solution notebook is available through a separate Moodle activity.


## C1: Shift and reverse with an explicit origin *(↔ P3, P4)*

A signal is stored as an array of values plus an `origin`: the array index that corresponds to $n=0$. The helper `times` returns the time index of every array entry. The signal is P3's $\{x[n]\} = \{\underset{\uparrow}{4},\,0,\,-2,\,1\}$.

a) Implement `delay(x, origin, k)`, returning the representation of $x[n-k]$. Hint: a delay does not change the values; it only moves the position of $n=0$.

b) Implement `reverse(x, origin)`, returning the representation of $x[-n]$. Hint: the origin is counted from the other end of the array.

c) Before running the check cell, write down (or take from your P3 and P4 solutions) the value at each time index for $x[n-1]$, $x[n+2]$, and $x[-n]$.

d) Run the check cell (it compares your functions against exactly those sequences), then run the plot cell. Compare the delayed and advanced signals with P4 and the reversed signal with P3.


In [ ]:
import numpy as np

x, origin = np.array([4, 0, -2, 1]), 0      # x = {4 (n=0), 0, -2, 1}

def times(x, origin):
    return np.arange(len(x)) - origin        # the time index of every array entry

def delay(x, origin, k):
    """Return (values, origin) representing x[n-k]."""
    ...  # your code here

def reverse(x, origin):
    """Return (values, origin) representing x[-n]."""
    ...  # your code here

In [ ]:
def as_dict(v, o):
    return {int(n): int(val) for n, val in zip(times(v, o), v)}

assert as_dict(*delay(x, origin, 1))  == {1: 4, 2: 0, 3: -2, 4: 1}
assert as_dict(*delay(x, origin, -2)) == {-2: 4, -1: 0, 0: -2, 1: 1}
assert as_dict(*reverse(x, origin))   == {-3: 1, -2: -2, -1: 0, 0: 4}
print("C1 passed: matches P3 and P4.")

In [ ]:
import matplotlib.pyplot as plt

signals = [
    ((x, origin), "$x[n]$"),
    (delay(x, origin, 1), "$x[n-1]$"),
    (delay(x, origin, -2), "$x[n+2]$"),
    (reverse(x, origin), "$x[-n]$"),
]

fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex=True, sharey=True)
for ax, ((v, o), title) in zip(axes.flat, signals):
    ax.stem(times(v, o), v)
    ax.set_title(title)
    ax.set_xlabel("$n$")
    ax.set_xlim(-4, 5)

plt.tight_layout()


## C2: Impulse decomposition, verified by reconstruction *(↔ P2)*

P2's signal is given by its nonzero samples only, as `(index, value)` pairs: $x[-1]=3$, $x[0]=1$, $x[2]=2$. The helper `delta(m)` is the impulse $\delta[n-m]$ on the grid `n`.

a) Build `x_rebuilt` as a sum of scaled, shifted impulses: one term `value * delta(index)` per pair: this is the correct option of P2, written in code.

b) Predict the values of `x_rebuilt` at $n=-1$, $n=0$, and $n=2$.

c) Run the check cell: it compares the reconstruction against the signal's direct sample list.

In [ ]:
pairs = [(-1, 3), (0, 1), (2, 2)]            # (index, value) of each nonzero sample

n = np.arange(-3, 5)
delta = lambda m: (n == m).astype(float)     # the impulse delta[n - m] on this grid

x_rebuilt = ...  # your code: sum of value * delta(index) over the pairs

In [ ]:
x_direct = np.array([0, 0, 3, 1, 0, 2, 0, 0], dtype=float)
assert np.allclose(x_rebuilt, x_direct)
print("C2 passed: the impulse sum IS the signal.")

## C3: Test a system for linearity and time invariance *(↔ P6, P10)*

Two systems are given: `switch_on`, which is P10 b)'s switched-on channel $y[n]=u[n]\,s[n]$, and `add_one`, which is $y[n]=u[n]+1$ (the constant-offset system from the lecture's nonlinear examples); also given are two random test inputs and a `shift` that delays by $k$ samples.

a) Implement `passes_linearity_test(sys)`: feed $\alpha u_1 + \beta u_2$ through the system and compare the result with $\alpha\,\mathsf{G}\{u_1\} + \beta\,\mathsf{G}\{u_2\}$: the linearity definition, exactly as in P10 a).

b) Implement `passes_time_invariance_test(sys)`: feed the delayed input `shift(u1, k)` and compare with the delayed output `shift(sys(u1), k)`. **Boundary caveat:** on a finite window, shifting pads with zeros, which silently assumes the signal is zero outside the window. Compare only the samples with index $\ge k$, where the delayed input's history lies fully inside the window.

c) Predict the four results (for each of the two systems: linear or not, time-invariant or not), then run the check cell.

*A passed random test suggests a property; only a derivation proves it. A single failed test, however, disproves it: the same asymmetry as in P10.*


In [ ]:
N = 16
s = (np.arange(N) - 4 >= 0).astype(float)    # unit step (origin at index 4)

switch_on = lambda u: u * s                  # P10 b): the switched-on channel y[n] = u[n] s[n]
add_one   = lambda u: u + 1                  # y[n] = u[n] + 1: adds a constant offset (not linear; why?)

def shift(u, k):                             # delay by k >= 0 samples
    v = np.zeros_like(u); v[k:] = u[:len(u) - k]; return v

rng = np.random.default_rng(0)
u1, u2 = rng.normal(size=N), rng.normal(size=N)

def passes_linearity_test(sys, a=2.0, b=-3.0):
    ...  # your code: one boolean

def passes_time_invariance_test(sys, k=3):
    ...  # your code: one boolean; compare only samples with index >= k


In [ ]:
results = {name: (passes_linearity_test(sys), passes_time_invariance_test(sys))
           for name, sys in [("y[n] = u[n] s[n]", switch_on), ("y[n] = u[n] + 1", add_one)]}
for name, (lin, ti) in results.items():
    print(f"{name}:  linear={lin}, time-invariant={ti}")
assert results["y[n] = u[n] s[n]"] == (True, False)   # linear, not TI  (P10)
assert results["y[n] = u[n] + 1"]  == (False, True)   # not linear, TI
print("C3 passed: matches P6/P10.")
